In [2]:
import pandas as pd
import numpy as np

## Discretization and Binning with `pd.cut`

Continuous data is often discretized into discrete **bins** for analysis.

`pd.cut` assigns each value to a bin defined by explicit edge boundaries.

### Output

Returns a **Categorical** object where:

- `.codes` — integer array mapping each value to its bin index.
- `.categories` — the `IntervalIndex` of all bins.
- `pd.value_counts(result)` — count of values in each bin.

### Interval Notation

| Symbol | Meaning |
|--------|---------|
| `(` or `)` | Open (exclusive) |
| `[` or `]` | Closed (inclusive) |

By default the **right** side of each interval is closed: `(18, 25]`.
Pass `right=False` to close the **left** side instead: `[18, 25)`.

### Options

| Argument | Effect |
|----------|--------|
| `bins` (list) | Explicit bin edges |
| `bins` (int) | Number of equal-length bins computed from min/max |
| `right=False` | Left-closed intervals instead of right-closed |
| `labels=[...]` | Custom string labels instead of interval notation |
| `precision=n` | Decimal precision for auto-computed bin edges |

## Key Points

- `pd.cut` assigns values to bins defined by explicit edges or an integer count.
- Returns a Categorical — use `.codes`, `.categories`, and `pd.value_counts` to inspect it.
- Right side is closed by default; `right=False` switches to left-closed.
- `labels=` replaces interval notation with custom names.
- Passing an integer creates equal-width bins based on the data's range.

In [3]:
ages = [20, 22, 25, 27, 21, 23, 37, 31, 61, 45, 41, 32]
bins = [18, 25, 35, 60, 100]

age_categories = pd.cut(ages, bins)

age_categories

[(18, 25], (18, 25], (18, 25], (25, 35], (18, 25], ..., (25, 35], (60, 100], (35, 60], (35, 60], (25, 35]]
Length: 12
Categories (4, interval[int64, right]): [(18, 25] < (25, 35] < (35, 60] < (60, 100]]

In [4]:
age_categories.codes  # integer bin index per value

array([0, 0, 0, 1, 0, 0, 2, 1, 3, 2, 2, 1], dtype=int8)

In [5]:
age_categories.categories  # the bin intervals

IntervalIndex([(18, 25], (25, 35], (35, 60], (60, 100]], dtype='interval[int64, right]')

In [6]:
age_categories.categories[0]  # inspect a single interval

Interval(18, 25, closed='right')

In [7]:
pd.value_counts(age_categories)  # count of values per bin

/tmp/ipykernel_16082/2546394378.py:1: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  pd.value_counts(age_categories)  # count of values per bin


(18, 25]     5
(25, 35]     3
(35, 60]     3
(60, 100]    1
Name: count, dtype: int64

In [8]:
pd.cut(ages, bins, right=False)  # left-closed intervals: [18, 25)

[[18, 25), [18, 25), [25, 35), [25, 35), [18, 25), ..., [25, 35), [60, 100), [35, 60), [35, 60), [25, 35)]
Length: 12
Categories (4, interval[int64, left]): [[18, 25) < [25, 35) < [35, 60) < [60, 100)]

In [9]:
group_names = ["Youth", "YoungAdult", "MiddleAged", "Senior"]

pd.cut(ages, bins, labels=group_names)

['Youth', 'Youth', 'Youth', 'YoungAdult', 'Youth', ..., 'YoungAdult', 'Senior', 'MiddleAged', 'MiddleAged', 'YoungAdult']
Length: 12
Categories (4, object): ['Youth' < 'YoungAdult' < 'MiddleAged' < 'Senior']

In [10]:
# Integer bins: equal-length ranges computed from min/max
data = np.random.uniform(size=20)

pd.cut(data, 4, precision=2)

[(0.032, 0.27], (0.51, 0.75], (0.27, 0.51], (0.51, 0.75], (0.75, 0.99], ..., (0.032, 0.27], (0.51, 0.75], (0.27, 0.51], (0.51, 0.75], (0.51, 0.75]]
Length: 20
Categories (4, interval[float64, right]): [(0.032, 0.27] < (0.27, 0.51] < (0.51, 0.75] < (0.75, 0.99]]

## Quantile-Based Binning with `pd.qcut`

`pd.qcut` bins data based on **sample quantiles** rather than equal-width ranges.

- `pd.cut` → equal-width bins (same range, unequal counts).
- `pd.qcut` → equal-count bins (same number of data points in each bin).

### Options

| Argument | Effect |
|----------|--------|
| `q` (int) | Number of quantile-based bins (e.g. `4` → quartiles) |
| `q` (list) | Explicit quantile boundaries between 0 and 1 |
| `precision=n` | Decimal precision for computed bin edges |

### When to Use Each

- Use `cut` when the **range** of each bin matters (e.g. defined age groups).
- Use `qcut` when **equal group sizes** matter (e.g. percentile-based segmentation).

## Key Points

- `qcut` produces roughly equal-sized groups, unlike `cut` which produces equal-width ranges.
- Pass an integer for standard quantile splits (4 → quartiles, 10 → deciles).
- Pass a list of quantile boundaries for custom splits (e.g. `[0, 0.1, 0.5, 0.9, 1.]`).
- Both `cut` and `qcut` return a Categorical and integrate well with groupby operations.

In [11]:
data = np.random.standard_normal(1000)

quartiles = pd.qcut(data, 4, precision=2)

quartiles

[(-0.68, 0.02], (-3.71, -0.68], (0.02, 0.72], (0.72, 3.01], (-3.71, -0.68], ..., (0.02, 0.72], (-3.71, -0.68], (0.72, 3.01], (-3.71, -0.68], (-0.68, 0.02]]
Length: 1000
Categories (4, interval[float64, right]): [(-3.71, -0.68] < (-0.68, 0.02] < (0.02, 0.72] < (0.72, 3.01]]

In [12]:
pd.value_counts(quartiles)  # each bin has ~250 values

/tmp/ipykernel_16082/2054772363.py:1: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  pd.value_counts(quartiles)  # each bin has ~250 values


(-3.71, -0.68]    250
(-0.68, 0.02]     250
(0.02, 0.72]      250
(0.72, 3.01]      250
Name: count, dtype: int64

In [13]:
# Custom quantile boundaries
pd.qcut(data, [0, 0.1, 0.5, 0.9, 1.]).value_counts()

(-3.703, -1.236]    100
(-1.236, 0.0203]    400
(0.0203, 1.276]     400
(1.276, 3.014]      100
Name: count, dtype: int64